# 设计与平台调用
先核验原32条采样，再枚举4种受限组合和导出AF3真实输入。此部分可离线运行。新的ProteinMPNN推理为独立JAX环境可选步骤，默认关闭。

In [ ]:
from pathlib import Path
import sys, subprocess, csv, tempfile
root = Path.cwd().resolve()
if not (root / "predict.py").is_file():
    root = root.parent
assert (root / "predict.py").is_file(), "Open this notebook inside project or project/notebooks"
print("Project:", root)


In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as d:
    for mode in ["archive", "enumerate"]:
        target = Path(d) / mode
        subprocess.run([sys.executable, "-B", str(root / "design.py"), "--mode", mode, "--out", str(target)], cwd=root, check=True)
        print(mode, [p.name for p in target.iterdir()])
    target = Path(d) / "requests"
    subprocess.run([sys.executable, "-B", str(root / "prepare_af3.py"), "--out", str(target)], cwd=root, check=True)
    print("AF3 request JSON files:", len(list(target.glob("*.json"))))


## 可选新AI设计
在独立配置的ColabDesign v1.1.1环境启用下一格；本地CPU复核不需安装JAX。默认不启用、不伪称新设计已执行。

In [ ]:
RUN_NEW_MPNN = False
if RUN_NEW_MPNN:
    subprocess.run([sys.executable, "-B", str(root / "design.py"), "--mode", "mpnn", "--seed", "20261003", "--out", str(root / "results/new_mpnn_notebook")], cwd=root, check=True)
else:
    print("Fresh MPNN skipped explicitly; original seed was not recorded.")


AF3任务需手动上传并下载ZIP，不能由此Notebook保证服务版本和输出逐值一致。导出的JSON来自实际任务，并不发起服务调用。